# Edge Effects Analysis

Loads the WDPA-level parquet, runs summary statistics, ANOVA, mixed models, and generates all publication figures.  
Index is read from `modules/config.py` (`INDEX_NAME`).

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()))
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from modules import config
from modules.analysis import recategorize_biome
from modules.plotting import classify_trend, create_correlation_plot, create_distribution_plots

IDX = config.INDEX_NAME
FIG_DIR = config.RESULTS_FIGURES
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_txt(name, text):
    path = FIG_DIR / f'{IDX}_{name}.txt'
    path.write_text(text)
    print(f'Saved {path}')

def save_fig(name, fig=None, **kw):
    path = FIG_DIR / f'{IDX}_{name}.png'
    (fig or plt.gcf()).savefig(path, dpi=300, bbox_inches='tight', **kw)
    plt.close('all')
    print(f'Saved {path}')

print(f'Index: {IDX.upper()}  |  {config.INDEX_CONFIGS[IDX]["description"]}')

Index: NDVI  |  Normalized Difference Vegetation Index


## 1 — Load & Prepare Data

In [2]:
wdpa_df = pd.read_parquet(config.RESULTS_DIR / f'wdpa_df_{IDX}.parquet')
wdpa_df['BIOME_NAME'] = wdpa_df['BIOME_NAME'].apply(recategorize_biome)

# Classify temporal trends for both metrics
for metric in ['edge_extent', 'edge_intensity']:
    col = f'trend_{metric.split("_")[1]}'
    trends = (wdpa_df.groupby('WDPA_PID')
              .apply(classify_trend, metric=metric)
              .rename(col).reset_index())
    wdpa_df = wdpa_df.merge(trends, on='WDPA_PID', how='left')

print(f'Rows: {len(wdpa_df):,}  |  PAs: {wdpa_df["WDPA_PID"].nunique():,}')
wdpa_df.head()

Rows: 93,036  |  PAs: 4,046


,WDPA_PID,year,n_trnst,D02,D01,D0m1,D0m2,edge_intensity,edge_extent,gHM_mean,...,SUB_LOC,SUPP_INFO,VERIF,WDPAID,geometry_t,AREA_DISSO,PERIMETER,PA_RATIO,trend_extent,trend_intensity
0,100017,2003,865,0.228921,0.172160,0.596666,0.731519,0.172160,0.605780,0.182738,...,KZ-ALM,Not Applicable,State Verified,100017.0,Polygon,3.117771e+09,582579.791312,0.000187,decrease,decrease
1,100017,2004,857,0.219684,0.197429,0.677738,0.770720,0.197429,0.597433,0.182738,...,KZ-ALM,Not Applicable,State Verified,100017.0,Polygon,3.117771e+09,582579.791312,0.000187,decrease,decrease
2,100017,2005,865,0.252335,0.232848,0.734410,0.769700,0.232848,0.604624,0.182738,...,KZ-ALM,Not Applicable,State Verified,100017.0,Polygon,3.117771e+09,582579.791312,0.000187,decrease,decrease
3,100017,2006,865,0.252208,0.266090,0.701824,0.762234,0.252208,0.625434,0.182738,...,KZ-ALM,Not Applicable,State Verified,100017.0,Polygon,3.117771e+09,582579.791312,0.000187,decrease,decrease
4,100017,2007,848,0.243323,0.208690,0.705725,0.780912,0.208690,0.614387,0.182738,...,KZ-ALM,Not Applicable,State Verified,100017.0,Polygon,3.117771e+09,582579.791312,0.000187,decrease,decrease


## 2 — Summary Statistics

In [3]:
pa = wdpa_df.groupby('WDPA_PID').first()  # one row per PA for counts
pa_mean = wdpa_df.groupby('WDPA_PID')[['edge_extent','edge_intensity']].mean()

lines = [
    f'Unique PAs: {len(pa):,}',
    f'Total transects: {pa["n_trnst"].sum():,.0f}',
    f'Countries (ISO3): {wdpa_df["ISO3"].nunique()}',
    f'Biomes: {wdpa_df["BIOME_NAME"].nunique()}',
    '',
    'PAs per biome:',
    *[f'  {b}: {c:,}' for b, c in pa['BIOME_NAME'].value_counts().items()],
    '',
    'Trend (edge_extent):',
    *[f'  {t}: {n:,}' for t, n in pa['trend_extent'].value_counts().items()],
    '',
    f'edge_extent < 10%: {(pa_mean["edge_extent"]<0.1).sum():,} '
    f'({(pa_mean["edge_extent"]<0.1).mean()*100:.1f}%)',
    f'edge_intensity < 0: {(pa_mean["edge_intensity"]<0).sum():,} '
    f'({(pa_mean["edge_intensity"]<0).mean()*100:.1f}%)',
]
summary_txt = '\n'.join(lines)
print(summary_txt)
save_txt('summary_statistics', summary_txt)

Unique PAs: 4,046
Total transects: 1,736,434
Countries (ISO3): 146
Biomes: 8

PAs per biome:
  Grassland & Shrubland: 1,307
  Tropical Forest: 1,166
  Temperate Forest: 712
  Boreal Forest: 371
  Desert: 358
  Tundra: 101
  Mangrove: 14
  Rock & Ice: 10

Trend (edge_extent):
  increase: 1,665
  decrease: 1,501
  sig_increase: 491
  sig_decrease: 388
  no_change: 1

edge_extent < 10%: 1 (0.0%)
edge_intensity < 0: 2,701 (66.8%)
Saved /workspace/results/figures/ndvi_summary_statistics.txt


## 3 — ANOVA

In [4]:
anova_df = wdpa_df[['edge_extent','IUCN_CAT','STATUS_YR','BIOME_NAME','AREA_DISSO']].dropna()

anova_results = []
for factor, formula in [
    ('IUCN_CAT',  'edge_extent ~ C(IUCN_CAT)'),
    ('STATUS_YR',  'edge_extent ~ STATUS_YR'),
    ('BIOME_NAME', 'edge_extent ~ C(BIOME_NAME)'),
    ('AREA_DISSO', 'edge_extent ~ AREA_DISSO'),
]:
    res = anova_lm(ols(formula, data=anova_df).fit(), typ=2)
    header = f'ANOVA for {factor}:\n{res}\n'
    print(header)
    anova_results.append(header)

save_txt('anova_results', '\n'.join(anova_results))

ANOVA for IUCN_CAT:
                 sum_sq       df         F         PR(>F)
C(IUCN_CAT)    6.046756      9.0  97.85566  6.858024e-183
Residual     637.596650  92865.0       NaN            NaN

ANOVA for STATUS_YR:
               sum_sq       df          F        PR(>F)
STATUS_YR    0.539516      1.0  77.913437  1.094748e-18
Residual   643.103890  92873.0        NaN           NaN

ANOVA for BIOME_NAME:
                   sum_sq       df           F  PR(>F)
C(BIOME_NAME)   16.168319      7.0  341.846985     0.0
Residual       627.475086  92867.0         NaN     NaN

ANOVA for AREA_DISSO:
                sum_sq       df         F        PR(>F)
AREA_DISSO    1.223612      1.0  176.8945  2.514527e-40
Residual    642.419793  92873.0       NaN           NaN

Saved /workspace/results/figures/ndvi_anova_results.txt


## 4 — Mixed-Effects Models

In [5]:

# Question #1: What is the trend in edge extent over time, and does it differ by biome?
# edge_extent ~ year_z * C(BIOME_NAME) + (1|WDPA_PID)

exclude_biomes = ['Mangrove', 'Rock & Ice']
df_q1 = (wdpa_df[~wdpa_df['BIOME_NAME'].isin(exclude_biomes)]
         [['edge_extent', 'year', 'BIOME_NAME', 'WDPA_PID']].dropna().copy())
df_q1['year_z'] = (df_q1['year'] - df_q1['year'].mean()) / df_q1['year'].std()

mdf_trend = MixedLM.from_formula(
    'edge_extent ~ year_z * C(BIOME_NAME)',
    data=df_q1, groups=df_q1['WDPA_PID'], re_formula='1'
).fit()
save_txt('edge_extent_trend_model', f"{'='*60}\nMIXED MODEL: TREND x BIOME\n{'='*60}\n\n{mdf_trend.summary()}")
print(mdf_trend.summary())

# --- Plot: slope (rate of change) per biome with 95% CI ---
fe = mdf_trend.fe_params
cov = mdf_trend.cov_params()
ref_biome = pd.Categorical(df_q1['BIOME_NAME']).categories[0]
biomes = sorted(df_q1['BIOME_NAME'].unique())

rows = []
for b in biomes:
    if b == ref_biome:
        slope = fe['year_z']
        se = np.sqrt(cov.loc['year_z', 'year_z'])
    else:
        int_key = f'year_z:C(BIOME_NAME)[T.{b}]'
        slope = fe['year_z'] + fe[int_key]
        se = np.sqrt(cov.loc['year_z', 'year_z']
                     + cov.loc[int_key, int_key]
                     + 2 * cov.loc['year_z', int_key])
    rows.append({'biome': b, 'slope': slope, 'lo': slope - 1.96*se, 'hi': slope + 1.96*se})

slopes = pd.DataFrame(rows).sort_values('slope')

fig, ax = plt.subplots(figsize=(10, 7))
y = range(len(slopes))
ax.errorbar(slopes['slope'], y,
            xerr=[slopes['slope'] - slopes['lo'], slopes['hi'] - slopes['slope']],
            fmt='o', capsize=5, color='#333')
ax.axvline(0, color='red', ls='--', lw=1)
ax.set_yticks(list(y))
ax.set_yticklabels(slopes['biome'])
ax.set_xlabel('Rate of Change in Edge Extent (per SD year)')
ax.set_title('Biome-Specific Temporal Trends in Edge Extent')
plt.tight_layout()
save_fig('figure3b_rateofchange_by_biome')


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Saved /workspace/results/figures/ndvi_edge_extent_trend_model.txt
                          Mixed Linear Model Regression Results
Model:                        MixedLM           Dependent Variable:           edge_extent
No. Observations:             92345             Method:                       REML       
No. Groups:                   4015              Scale:                        0.0014     
Min. group size:              23                Log-Likelihood:               161944.3406
Max. group size:              23                Converged:                    Yes        
Mean group size:              23.0                                                       
-----------------------------------------------------------------------------------------
                                              Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------------
Intercept                                      0.534    0.00

In [6]:
# Question # 2: What is influencing the edge extent? 
# edge_extent_2025 ~ z_covariates + + (1|WDPA_PID)

PREDICTORS = ['STATUS_YR','AREA_DISSO','gHM_mean','elevation_mean','slope_mean','water_extent_pct']
KEEP_COLS = PREDICTORS + ['BIOME_NAME','WDPA_PID']

def fit_mixed(response):
    """Fit MixedLM for *response* ~ z-scored predictors + (1|WDPA_PID)."""
    df_filtered = wdpa_df[wdpa_df['year'] == 2025].copy()
    df = df_filtered[[response] + KEEP_COLS].dropna().copy()
    for c in PREDICTORS:
        df[f'{c}_z'] = (df[c] - df[c].mean()) / df[c].std()
    formula = f"{response} ~ " + " + ".join(f'{c}_z' for c in PREDICTORS)
    mdf = MixedLM.from_formula(formula, data=df, groups=df['WDPA_PID'],
                               re_formula='1').fit()
    return mdf, df

models = {}
for resp, label, stem in [
    ('edge_extent',    'EDGE EXTENT',    'edge_extent_model'),
    ('edge_intensity', 'EDGE INTENSITY', 'edge_intensity_model'),
]:
    mdf, mdf_data = fit_mixed(resp)
    models[resp] = (mdf, mdf_data)
    save_txt(stem, f"{'='*60}\nMIXED MODEL: {label}\n{'='*60}\n\n{mdf.summary()}")
    print(mdf.summary(), '\n')


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


Saved /workspace/results/figures/ndvi_edge_extent_model.txt
            Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  edge_extent
No. Observations:    3785     Method:              REML       
No. Groups:          3785     Scale:               0.0033     
Min. group size:     1        Log-Likelihood:      4097.9229  
Max. group size:     1        Converged:           Yes        
Mean group size:     1.0                                      
--------------------------------------------------------------
                   Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept           0.555    0.001 420.724 0.000  0.553  0.558
STATUS_YR_z         0.005    0.001   3.406 0.001  0.002  0.007
AREA_DISSO_z        0.003    0.001   2.348 0.019  0.001  0.006
gHM_mean_z          0.005    0.001   3.475 0.001  0.002  0.008
elevation_mean_z   -0.006    0.002  -3.434 0.001 -0.010 -0.003
slope_me

/usr/local/lib/python3.10/dist-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


### Diagnostics (VIF, residuals, observed vs fitted)

In [7]:
for resp, (mdf, mdf_data) in models.items():
    stem = f'edge_{resp.split("_")[1]}_model'
    z_cols = [f'{c}_z' for c in PREDICTORS]

    # VIF
    X = mdf_data[z_cols]
    vif = pd.DataFrame({'variable': z_cols,
                        'VIF': [variance_inflation_factor(X.values, i) for i in range(len(z_cols))]})
    print(f'--- {resp} VIF ---'); print(vif, '\n')

    fitted, observed = mdf.fittedvalues, mdf_data[resp]
    resid = observed.values - fitted.values

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(fitted, observed, alpha=.3, s=5)
    lims = [observed.min(), observed.max()]
    axes[0].plot(lims, lims, 'r--')
    axes[0].set(xlabel='Fitted', ylabel='Observed', title=f'Obs v Fitted: {resp}')
    axes[1].hist(resid, bins=50, edgecolor='k')
    axes[1].set(xlabel='Residual', ylabel='Freq', title=f'Residuals: {resp}')
    plt.tight_layout()
    save_fig(f'{stem}_diagnostics')

--- edge_extent VIF ---
             variable       VIF
0         STATUS_YR_z  1.074798
1        AREA_DISSO_z  1.057298
2          gHM_mean_z  1.106759
3    elevation_mean_z  1.876483
4        slope_mean_z  1.905077
5  water_extent_pct_z  1.057172 

Saved /workspace/results/figures/ndvi_edge_extent_model_diagnostics.png
--- edge_intensity VIF ---
             variable       VIF
0         STATUS_YR_z  1.074798
1        AREA_DISSO_z  1.057298
2          gHM_mean_z  1.106759
3    elevation_mean_z  1.876483
4        slope_mean_z  1.905077
5  water_extent_pct_z  1.057172 

Saved /workspace/results/figures/ndvi_edge_intensity_model_diagnostics.png


## 5 — Figures

### Figure 2 — Map of Edge-Extent

In [8]:

import geopandas as gpd
from matplotlib.patches import Patch
from matplotlib.colors import TwoSlopeNorm

# Load PA geometries and join trend data
gdf = gpd.read_file(config.DATA_INTERMEDIATE / 'wdpa_filtered' / 'wdpa_filtered.shp')
gdf['WDPA_PID'] = gdf['WDPA_PID'].astype(str)
trend_per_pa = wdpa_df.groupby('WDPA_PID')['trend_extent'].first().reset_index()
trend_per_pa['WDPA_PID'] = trend_per_pa['WDPA_PID'].astype(str)
gdf = gdf.merge(trend_per_pa, on='WDPA_PID', how='inner')
gdf_moll = gdf.to_crs('+proj=moll')

# Collapse to 3 categories
gdf_moll['trend_extent'] = gdf_moll['trend_extent'].replace(
    {'increase': 'not_significant', 'decrease': 'not_significant', 'no_change': 'not_significant'})
cat_order = ['sig_increase', 'not_significant', 'sig_decrease']
cat_labels = {'sig_increase': 'Significant Increase', 'not_significant': 'Not Significant',
              'sig_decrease': 'Significant Decrease'}
colors = {'sig_increase': '#d73027', 'not_significant': '#afab9e', 'sig_decrease': '#4575b4'}
gdf_moll['trend_extent'] = pd.Categorical(gdf_moll['trend_extent'], categories=cat_order, ordered=True)

# World outline
world = gpd.read_file(
    'https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip'
).to_crs('+proj=moll')

fig, ax = plt.subplots(figsize=(16, 9))
world.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.3)
for cat in cat_order:
    subset = gdf_moll[gdf_moll['trend_extent'] == cat]
    if len(subset):
        subset.plot(ax=ax, color=colors[cat], markersize=4)
ax.set_axis_off()
handles = [Patch(facecolor=colors[c], label=cat_labels[c]) for c in cat_order]
ax.legend(handles=handles, title='Change in Edge Extent (2003–2025)',
          loc='lower left', frameon=True, fontsize=9)
ax.set_title('Figure 2: Edge Extent Trends by Protected Area', fontsize=14)
plt.tight_layout()
save_fig('figure2a_trend_map')


Saved /workspace/results/figures/ndvi_figure2a_trend_map.png


In [9]:
# Figure 2b: Edge Extent in 2025 (coolwarm, bent at median)
extent_2025 = (wdpa_df.loc[wdpa_df['year'] == 2025, ['WDPA_PID', 'edge_extent']]
               .groupby('WDPA_PID')['edge_extent'].mean().reset_index())
extent_2025['WDPA_PID'] = extent_2025['WDPA_PID'].astype(str)
gdf_2025 = gdf_moll.merge(extent_2025, on='WDPA_PID', how='inner')

fig, ax = plt.subplots(figsize=(16, 9))
world.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.3)
norm = TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
gdf_2025.plot(ax=ax, column='edge_extent', cmap='coolwarm', norm=norm,
              markersize=4, legend=True,
              legend_kwds={'label': 'Edge Extent (2025)', 'shrink': 0.5, 'pad': 0.02})
ax.set_axis_off()
ax.set_title('Figure 2b: Edge Extent in 2025', fontsize=14)
plt.tight_layout()
save_fig('figure2b_2025extent_map')


Saved /workspace/results/figures/ndvi_figure2b_2025extent_map.png


### Figure 3 — Edge-Extent Trends by Biome

In [10]:
exclude = ['Mangroves', 'Rock & Ice']
trend_by_biome = (wdpa_df[~wdpa_df['BIOME_NAME'].isin(exclude)]
                  .groupby(['BIOME_NAME','WDPA_PID'])['trend_extent']
                  .first().reset_index())
counts = (trend_by_biome.groupby(['BIOME_NAME','trend_extent']).size()
          .unstack(fill_value=0))
col_order = ['sig_decrease','decrease','increase','sig_increase']
counts = counts[[c for c in col_order if c in counts.columns]]
pcts = counts.div(counts.sum(axis=1), axis=0) * 100
totals = counts.sum(axis=1)

# Sort by sig_increase % descending (highest on top in barh = last row)
pcts = pcts.sort_values('sig_increase', ascending=True)
totals = totals.reindex(pcts.index)

fig, ax = plt.subplots(figsize=(10, 8))
pcts.plot(kind='barh', stacked=True, ax=ax,
          color=['#4575b4','#91bfdb','#fc8d59','#d73027'])
ax.set_yticklabels([f'{b} (n={totals[b]})' for b in pcts.index])
for i, biome in enumerate(pcts.index):
    cum = 0
    for col in pcts.columns:
        v = pcts.loc[biome, col]
        if v > 5:
            ax.text(cum + v/2, i, f'{v:.1f}%', ha='center', va='center', fontsize=8)
        cum += v
ax.set(xlabel='Percentage of Protected Areas', ylabel='Biome')
cat_labels = {'sig_increase': 'Significant Increase', 'increase': 'Increase',
              'decrease': 'Decrease', 'sig_decrease': 'Significant Decrease'}
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [cat_labels.get(l, l) for l in labels],
          title='Trend', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_title('Edge Extent Trends by Biome')
plt.tight_layout()
save_fig('figure3a_trends_by_biome')


Saved /workspace/results/figures/ndvi_figure3a_trends_by_biome.png


### Figure 4a/b — Mixed-Model Coefficients

In [11]:
for resp, fig_label in [('edge_extent', 'Figure 4a'), ('edge_intensity', 'Figure 4b')]:
    mdf = models[resp][0]
    ci = mdf.conf_int()
    coef = pd.DataFrame({'var': mdf.params.index, 'coef': mdf.params.values,
                         'lo': ci.iloc[:,0], 'hi': ci.iloc[:,1]})
    coef = coef[~coef['var'].isin(['Group Var','Intercept'])].reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.errorbar(coef['coef'], range(len(coef)),
                xerr=[coef['coef']-coef['lo'], coef['hi']-coef['coef']],
                fmt='o', capsize=5)
    ax.axvline(0, color='red', ls='--', lw=1)
    ax.set_yticks(range(len(coef)))
    ax.set_yticklabels(coef['var'])
    ax.set_xlabel('Coefficient (95% CI)')
    ax.set_title(f'{fig_label}: {resp.replace("_"," ").title()} ~ Covariates')
    plt.tight_layout()
    save_fig(f'figure4{"a" if resp=="edge_extent" else "b"}_{resp}_model')

Saved /workspace/results/figures/ndvi_figure4a_edge_extent_model.png
Saved /workspace/results/figures/ndvi_figure4b_edge_intensity_model.png


### Figure S1 — Distributions  &  Figure S2 — Correlation

In [12]:
# add gradient images of a few in the intensity-edge correlation plot 
# move correlation plot to main manusript figure, add gradient images and park boundary of 4-5 examples

df_filtered = wdpa_df[wdpa_df['year'] == 2025].copy()

create_distribution_plots(df_filtered, FIG_DIR / f'{IDX}_figureS1_distributions.png', IDX)
create_correlation_plot(df_filtered, FIG_DIR / f'{IDX}_figureS2_correlation.png', IDX)

Distribution plots saved to /workspace/results/figures/ndvi_figureS1_distributions.png
Correlation plot saved to /workspace/results/figures/ndvi_figureS2_correlation.png


---
All outputs saved to `results/figures/{INDEX_NAME}_*`.

## 6 — Export Data for Interactive Map
### Edge Extent Overtime

In [13]:
import glob

# Load boundary point coordinates (pointID == 0)
transects = pd.read_csv(config.RESULTS_DIR / 'processed' / 'transects_final.csv')
transects['pointID'] = transects['pointID'].astype(float)
coords = transects.loc[transects['pointID'] == 0.0, ['WDPA_PID', 'transectID', 'x', 'y']]
coords = coords.rename(columns={'x': 'lon', 'y': 'lat'})
coords['WDPA_PID'] = coords['WDPA_PID'].astype(str)
coords['transectID'] = coords['transectID'].astype(str)
print(f'Boundary coords loaded: {len(coords):,}')

# Load all transect chunks and join coordinates
chunk_dir = config.RESULTS_RAW / f'{IDX}_raw' / 'transect_chunks'
chunks = sorted(glob.glob(str(chunk_dir / f'{IDX}_transect_chunk_*.parquet')))

parts = []
for f in chunks:
    df = pd.read_parquet(f, columns=['pt_0', 'edge']).reset_index()
    df['WDPA_PID'] = df['WDPA_PID'].astype(str)
    df['transectID'] = df['transectID'].astype(str)
    df = df.merge(coords, on=['WDPA_PID', 'transectID'], how='inner')
    parts.append(df[['WDPA_PID', 'year', 'pt_0', 'edge', 'lon', 'lat']])
    print(f'  {Path(f).name}: {len(df):,} rows')

export = pd.concat(parts, ignore_index=True)
print(f'\nTotal rows: {len(export):,}  |  PAs: {export["WDPA_PID"].nunique():,}')

out_path = config.RESULTS_DIR / f'{IDX}_boundary_points_yearly.csv'
export.to_csv(out_path, index=False)
print(f'Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)')

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/results/processed/transects_final.csv'